In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Загрузка данных из Excel-файла
data = pd.read_excel('synthetic_dataset_BETA_3.xlsx')

# Вычисляем целевую переменную: оптимальное количество закупок
# Если остаток меньше продаж, то необходимо докупить разницу, иначе 0
data['Optimal_Purchase_Quantity'] = (data['Quantity_Sold'] - data['Stock_Left']).apply(lambda x: max(x, 0))

# Обработка столбцов с датами: переводим их в числовое представление (ordinal)
date_cols = ['Start_Expiry_Date', 'End_Expiry_Date', 'Purchase_Date', 'Sales_Date']
for col in date_cols:
    data[col] = pd.to_datetime(data[col], errors='coerce')
    data[col] = data[col].apply(lambda x: x.toordinal() if pd.notnull(x) else np.nan)

# Заполним пропуски в числовых столбцах медианными значениями
data = data.fillna(data.median(numeric_only=True))

# Определяем признаки (X) и целевую переменную (y)
X = data.drop(columns=['Optimal_Purchase_Quantity'])
y = data['Optimal_Purchase_Quantity']

# Разбиваем данные на обучающую и тестовую выборки (опционально)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Обучаем RandomForestRegressor для оценки важности признаков
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Получаем важность признаков
importances = rf.feature_importances_
feature_names = X.columns

# Создаём DataFrame для удобства отображения важности признаков
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# Выводим важность признаков в консоль
print(importance_df)

# Визуализируем важность признаков
plt.figure(figsize=(12, 8))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='skyblue')
plt.xlabel('Важность')
plt.title('Важность признаков по модели Random Forest')
plt.gca().invert_yaxis()  # Самые важные признаки сверху
plt.show()


ValueError: could not convert string to float: 'Asparagus'